#### Step 1 导入相关包

In [8]:
import pandas as pd
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel
import torch

import pickle

from sentence_transformers import SentenceTransformer

In [7]:
model = SentenceTransformer('AI-Growth-Lab/PatentSBERTa',device='cuda')

d:\anaconda3\envs\pytorch\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


#### Step 2 加载数据

In [5]:
df_abstract = pd.read_csv('./title_abstract.csv')

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18636\1662150031.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_abstract = pd.read_csv('./title_abstract.csv')


In [6]:
df_abstract

,patent_id,patent_abstract,patent_title
0,10000000,A frequency modulated (coherent) laser detecti...,Coherent LADAR using intra-pixel quadrature de...
1,10000001,The injection molding machine includes a fixed...,Injection molding machine and mold thickness c...
2,10000002,The present invention relates to: a method for...,Method for manufacturing polymer film and co-e...
3,10000003,The invention relates to a method for producin...,Method for producing a container from a thermo...
4,10000004,The present invention relates to provides a do...,"Process of obtaining a double-oriented film, c..."
...,...,...,...
8206410,RE50353,"Circuits for a voltage regulator are provided,...",Circuits and methods for hybrid 3:1 voltage re...
8206411,RE50354,A method of detecting patterns in network traf...,Automatic detection of malicious packets in DD...
8206412,RE50355,A first portion of programming aired prior to ...,Reducing unicast session duration with restart TV
8206413,RE50356,Embodiments described herein provide for a dis...,DAS management by radio access network node


#### Step 3 计算嵌入

In [9]:
texts = (df_abstract['patent_title'].fillna('') + ' ' + df_abstract['patent_abstract'].fillna('')).tolist()
ids   = df_abstract['patent_id'].tolist()

BATCH_SIZE = 50

In [14]:
id2vec = {}
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch_texts = texts[i : i+BATCH_SIZE]
    batch_ids   = ids[i : i+BATCH_SIZE]
    vecs = model.encode(batch_texts,
                        batch_size=BATCH_SIZE,
                        show_progress_bar=False,
                        convert_to_numpy=True)
    for pid, vec in zip(batch_ids, vecs):
        id2vec[pid] = vec

100%|██████████| 164129/164129 [7:12:11<00:00,  6.33it/s]  


In [16]:
len(id2vec)

8206415

#### Step 4 分成多个pickle存储

In [19]:
import pickle
import math

CHUNK_SIZE = 1000000
total_items = len(id2vec)
num_parts = math.ceil(total_items / CHUNK_SIZE)

items = list(id2vec.items())          # 转成列表以保证顺序

for idx in range(num_parts):
    start = idx * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, total_items)
    part_dict = dict(items[start:end])

    fname = f'./result/id2vec_part_{idx:03d}.pkl'
    with open(fname, 'wb') as f:
        pickle.dump(part_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'{fname} saved, records {start}-{end-1}')

print('All done!')

./result/id2vec_part_000.pkl saved, records 0-999999
./result/id2vec_part_001.pkl saved, records 1000000-1999999
./result/id2vec_part_002.pkl saved, records 2000000-2999999
./result/id2vec_part_003.pkl saved, records 3000000-3999999
./result/id2vec_part_004.pkl saved, records 4000000-4999999
./result/id2vec_part_005.pkl saved, records 5000000-5999999
./result/id2vec_part_006.pkl saved, records 6000000-6999999
./result/id2vec_part_007.pkl saved, records 7000000-7999999
./result/id2vec_part_008.pkl saved, records 8000000-8206414
All done!
